# Fraud Detection using Machine Learning  
### Internship Task Submission  

**Author:** Ankit Yadav  
**Dataset Size:** 6.3 Million Transactions  
**Objective:** Predict fraudulent financial transactions  

## 1. Business Problem

The objective of this project is to build a machine learning model to proactively detect fraudulent financial transactions in a financial system.

Fraud detection helps reduce financial losses and improves trust in digital payment systems.

## 2. Data Loading

The dataset contains 6.3 million financial transactions in CSV format. The data was loaded into a pandas DataFrame for further preprocessing and analysis.

In [48]:
df_full = pd.read_csv('Fraud.csv', low_memory=False)
df_full.shape

(6362620, 11)

## 3. Data Understanding

The dataset contains transaction-level information including transaction amount, account balances before and after transaction, and transaction type.

Key Variables:

- **step**: Time step (in hours)
- **type**: Type of transaction (PAYMENT, TRANSFER, CASH_OUT, etc.)
- **amount**: Transaction amount
- **oldbalanceOrg**: Sender's balance before transaction
- **newbalanceOrig**: Sender's balance after transaction
- **oldbalanceDest**: Receiver's balance before transaction
- **newbalanceDest**: Receiver's balance after transaction
- **isFraud**: Target variable (1 = Fraud, 0 = Normal)
- **isFlaggedFraud**: System flag for large transactions

In [49]:
df_full.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [50]:
df_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


## 4. Data Cleaning & Preprocessing

To prepare the dataset for modeling, the following steps were performed:

- Removed missing values
- Converted target variables to integer type
- Dropped unnecessary identifier columns
- Applied One-Hot Encoding to categorical variables

In [51]:
# Remove missing values
df_full = df_full.dropna()

# Convert target columns to integer
df_full['isFraud'] = df_full['isFraud'].astype(int)
df_full['isFlaggedFraud'] = df_full['isFlaggedFraud'].astype(int)

# Drop identifier columns
df_full = df_full.drop(['nameOrig', 'nameDest'], axis=1)

# One-Hot Encoding
df_full = pd.get_dummies(df_full, columns=['type'], drop_first=True)

df_full.shape

(6362620, 12)

## 5. Feature and Target Selection

The dataset was divided into input features (X) and the target variable (y).
The target variable is 'isFraud'.

In [52]:
X_full = df_full.drop('isFraud', axis=1)
y_full = df_full['isFraud']

## 6. Train-Test Split

The dataset was split into training and testing sets using stratified sampling to preserve fraud class distribution.

In [53]:
from sklearn.model_selection import train_test_split

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_full,
    y_full,
    test_size=0.2,
    random_state=42,
    stratify=y_full
)

## 7. Model Development – Random Forest

Random Forest was selected as the final model due to its ability to handle class imbalance and non-linear relationships effectively.

In [54]:
from sklearn.ensemble import RandomForestClassifier

rf_final = RandomForestClassifier(
    n_estimators=80,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

rf_final.fit(X_train_f, y_train_f)

RandomForestClassifier(class_weight='balanced', n_estimators=80, n_jobs=-1,
                       random_state=42)

## 8. Model Evaluation


The Random Forest model was evaluated using confusion matrix and classification metrics such as precision, recall, and F1-score.

The model achieved strong performance despite severe class imbalance in the dataset.

Fraud Detection Performance:
- Precision: 0.98
- Recall: 0.78
- F1-Score: 0.87

The use of class_weight="balanced" helped improve fraud detection while keeping false positives low.

In [55]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred_final = rf_final.predict(X_test_f)

print(confusion_matrix(y_test_f, y_pred_final))
print(classification_report(y_test_f, y_pred_final))

[[1270854      27]
 [    358    1285]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.98      0.78      0.87      1643

    accuracy                           1.00   1272524
   macro avg       0.99      0.89      0.93   1272524
weighted avg       1.00      1.00      1.00   1272524



## Feature Importance Analysis

Feature importance analysis helps identify which variables contribute most to fraud detection.

Account balance related variables appear to be the strongest predictors of fraudulent activity.

In [56]:
import pandas as pd

feature_importance = pd.DataFrame({
    'Feature': X_full.columns,
    'Importance': rf_final.feature_importances_
}).sort_values(by='Importance', ascending=False)

feature_importance.head(10)

,Feature,Importance
2,oldbalanceOrg,0.325649
1,amount,0.167125
3,newbalanceOrig,0.159072
10,type_TRANSFER,0.089624
9,type_PAYMENT,0.066197
5,newbalanceDest,0.056515
0,step,0.046600
7,type_CASH_OUT,0.046124
4,oldbalanceDest,0.042451
6,isFlaggedFraud,0.000428


## 9. Business Recommendations

Based on the model results and feature importance:

- Transactions with abnormal balance changes should be monitored closely.
- Large TRANSFER and CASH_OUT transactions require additional verification.
- Real-time fraud detection systems should prioritize high-risk transactions.
- Continuous retraining of the model is recommended to adapt to evolving fraud patterns.